# Logistic Regression Gradient & Convexity — AI Lab Instructor Notebook

*Classical ML · Medium*

Derive and implement logistic regression *correctly* (loss, gradient, Hessian intuition) and explain why the loss is convex.

**Outcome.** Students can implement vectorized logistic regression, gradient-check it, and communicate convexity + optimization reasoning at FAANG interview depth.

8 tasks, all with solutions.

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_data(n=400, d=5, separable=False):
    X = rng.standard_normal((n, d))
    w_true = rng.standard_normal(d)
    logits = X @ w_true
    if separable:
        y = (logits > 0).astype(int)
    else:
        logits = logits + 0.5 * rng.standard_normal(n)
        probs = 1 / (1 + np.exp(-logits))
        y = (rng.random(n) < probs).astype(int)
    return X, y, w_true

X, y, w_true = make_data(separable=False)

## Task 1: Dataset & Setup

# Logistic Regression Gradient & Convexity — Student Lab

Complete all TODOs. This lab is math-first and stability-first.

## Section 0 — Synthetic Dataset
We’ll create a binary classification dataset with both separable and non-separable regimes.

![Synthetic dataset (first two features)](/images/w3-d1-synthetic-first-two-features.svg)

In [ ]:
import numpy as np

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

def make_data(n=400, d=5, separable=False):
    X = rng.standard_normal((n, d))
    w_true = rng.standard_normal(d)
    logits = X @ w_true
    if separable:
        y = (logits > 0).astype(int)
    else:
        logits = logits + 0.5 * rng.standard_normal(n)
        probs = 1 / (1 + np.exp(-logits))
        y = (rng.random(n) < probs).astype(int)
    return X, y, w_true

X, y, w_true = make_data(separable=False)
check('shapes', X.ndim==2 and y.ndim==1 and X.shape[0]==y.shape[0])

**Why this works.** # Logistic Regression Gradient & Convexity — Instructor (Solutions + Rationale)
Runs top-to-bottom.

## Section 0 — Synthetic Dataset

Rationale: plotting the first two features gives a fast visual check of overlap and class balance before deriving gradients.

![Synthetic dataset (first two features)](/images/w3-d1-synthetic-first-two-features.svg)

## Task 2: Stable sigmoid
*Section: Sigmoid + Stable Loss*

## Section 1 — Sigmoid + Stable Loss

### Task 1.1: Stable sigmoid

**Explain:** Why does sigmoid saturate for large |z| and what does that do to gradients?

**Hints**
- # - Use `np.where(z>=0, ...)` trick to avoid overflow.

**Explain:** Why does sigmoid saturate for large |z| and what does that do to gradients?

In [ ]:
def sigmoid(z):
    z = np.asarray(z)
    return np.where(z >= 0, 1/(1+np.exp(-z)), np.exp(z)/(1+np.exp(z)))

z = np.array([-1000.0, -10.0, 0.0, 10.0, 1000.0])
p = sigmoid(z)
check('in_0_1', np.all((p >= 0) & (p <= 1)))
check('monotonic', np.all(np.diff(p) >= 0))

# Rationale: stable branching avoids overflow in exp for large negative logits.

In [ ]:
# Checks
check('in_0_1', np.all((p >= 0) & (p <= 1)))
check('monotonic', np.all(np.diff(p) >= 0))

**Why this works.** stable branching avoids overflow in exp for large negative logits.

## Section 1 — Stable sigmoid + loss

## Task 3: Stable binary cross-entropy loss
*Section: Sigmoid + Stable Loss*

### Task 1.2: Stable binary cross-entropy loss

We use labels y in {0,1}.

Loss per example: `-y log(p) - (1-y) log(1-p)` where p=sigmoid(z).

**Interview Angle:** explain a stable form of log-loss.

**Hints**
- # - Clip p to [eps, 1-eps]
- # - Alternatively use softplus: log(1+exp(z))

In [ ]:
def bce_loss(y, p):
    eps = 1e-12
    p = np.clip(p, eps, 1-eps)
    y = y.astype(float)
    return float(-np.mean(y* np.log(p) + (1-y)* np.log(1-p)))

L = bce_loss(y, sigmoid(X @ np.zeros(X.shape[1])))
check('finite_loss', np.isfinite(L))

In [ ]:
# Checks
check('finite_loss', np.isfinite(L))

**Why this works.** clipping prevents log(0) and keeps loss finite.

## Task 4: Implement loss and gradient
*Section: Gradient (Derive → Implement → Check)*

## Section 2 — Gradient (Derive → Implement → Check)

### Task 2.1: Derive the gradient (write in markdown)
Show that for loss averaged over n examples:
`grad = X^T (p - y) / n`

**Checkpoint:** What is the shape of grad?

### Task 2.2: Implement loss and gradient

**FAANG gotcha:** ensure y is 0/1, not -1/+1.

**Hints**
- # - z = X @ w
- # - p = sigmoid(z)
- # - loss = BCE
- # - grad = X.T @ (p - y) / n

In [ ]:
def logreg_loss_and_grad(X, y, w):
    z = X @ w
    p = sigmoid(z)
    loss = bce_loss(y, p)
    grad = (X.T @ (p - y)) / X.shape[0]
    return loss, grad

w0 = rng.standard_normal(X.shape[1])
loss, grad = logreg_loss_and_grad(X, y, w0)
print('loss', loss, 'grad_norm', np.linalg.norm(grad))
check('grad_shape', grad.shape == w0.shape)
check('finite', np.isfinite(loss) and np.all(np.isfinite(grad)))

# Rationale: gradient simplifies to X^T (p - y) / n by chain rule.

In [ ]:
# Checks
check('grad_shape', grad.shape == w0.shape)
check('finite', np.isfinite(loss) and np.all(np.isfinite(grad)))

**Why this works.** gradient simplifies to X^T (p - y) / n by chain rule.

## Section 2 — Loss + gradient + gradient check

## Task 5: Numerical gradient check (finite differences)
*Section: Gradient (Derive → Implement → Check)*

### Task 2.3: Numerical gradient check (finite differences)

Compare your analytic gradient to a central-difference numerical gradient.

**Checkpoint:** Why can eps too small make the check worse?

**Hints**
- central difference

In [ ]:
def numerical_grad(f, w, eps=1e-5):
    w = np.asarray(w, dtype=float)
    g = np.zeros_like(w)
    for i in range(w.size):
        e = np.zeros_like(w)
        e[i] = 1.0
        g[i] = (f(w + eps * e) - f(w - eps * e)) / (2 * eps)
    return g

f = lambda v: logreg_loss_and_grad(X, y, v)[0]
g_num = numerical_grad(f, w0)
_, g = logreg_loss_and_grad(X, y, w0)

max_abs = np.max(np.abs(g - g_num))
print('max_abs_diff', max_abs)
check('grad_check', max_abs < 1e-5)

# Rationale: finite differences validate the analytic gradient implementation.

In [ ]:
# Checks
check('grad_check', max_abs < 1e-5)

**Why this works.** finite differences validate the analytic gradient implementation.

## Task 6: Hessian-vector product (HVP)
*Section: Convexity & Hessian Intuition*

## Section 3 — Convexity & Hessian Intuition

### Task 3.1: Hessian-vector product (HVP)

For logistic regression:
H = (1/n) X^T S X where S = diag(p(1-p)).

Implement HVP: compute H@v without building full H explicitly.

- s = p*(1-p)
- compute Xv then multiply by s then X^T

**Hints**
- s = p*(1-p)
compute Xv then multiply by s then X^T

In [ ]:
def hvp(X, y, w, v):
    z = X @ w
    p = sigmoid(z)
    s = p * (1 - p)
    Xv = X @ v
    return (X.T @ (s * Xv)) / X.shape[0]

v = rng.standard_normal(X.shape[1])
Hv = hvp(X, y, w0, v)
check('hvp_shape', Hv.shape == v.shape)
check('hvp_finite', np.all(np.isfinite(Hv)))

# Rationale: this computes H@v directly without materializing the full Hessian.

In [ ]:
# Checks
check('hvp_shape', Hv.shape == v.shape)
check('hvp_finite', np.all(np.isfinite(Hv)))

**Why this works.** this computes H@v directly without materializing the full Hessian.

## Section 3 — Hessian-vector product + PSD

## Task 7: Empirical PSD check
*Section: Convexity & Hessian Intuition*

### Task 3.2: Empirical PSD check

Check that v^T H v >= 0 for random v (PSD).

**Explain:** Why does PSD imply convexity?

**Explain:** Why does PSD imply convexity?

In [ ]:
def vHv(X, y, w, v):
    return float(v @ hvp(X, y, w, v))

vals = []
for _ in range(50):
    v = rng.standard_normal(X.shape[1])
    vals.append(vHv(X, y, w0, v))
print('min vHv', min(vals))
check('psd', min(vals) > -1e-8)

# Rationale: v^T H v >= 0 for all v is the PSD condition, implying convexity.

In [ ]:
# Checks
check('psd', min(vals) > -1e-8)

**Why this works.** v^T H v >= 0 for all v is the PSD condition, implying convexity.

## Task 8: One step of GD vs Newton (conceptual)
*Section: Optimization (Bonus)*

## Section 4 — Optimization (Bonus)

### Task 4.1: One step of GD vs Newton (conceptual)

Implement one gradient descent step. (Newton step is optional.)

**FAANG gotcha:** perfectly separable data can push weights to infinity; explain why.

## Section 4 — GD Step

Gradient descent update rule:

`w' = w - η ∇L(w)`

Where:

- `w` = current weights
- `w'` = updated weights after one optimization step
- `η` = learning rate (step size)
- `∇L(w)` = gradient of the loss with respect to `w`

Newton (Hessian) step notation:

`w' = w - H(w)^{-1} ∇L(w)`

Where:

- `H(w) = ∇²L(w)` = Hessian matrix (second derivatives)
- `H(w)^{-1} ∇L(w)` rescales the gradient using local curvature

Interpretation: gradient descent uses only slope; Newton step uses slope + curvature for a more curvature-aware update.

In [ ]:
def gd_step(X, y, w, lr):
    _, g = logreg_loss_and_grad(X, y, w)
    return w - lr * g

w1 = gd_step(X, y, w0, lr=0.1)
loss0, _ = logreg_loss_and_grad(X, y, w0)
loss1, _ = logreg_loss_and_grad(X, y, w1)
print('loss0', loss0, 'loss1', loss1)
check('decreased', loss1 <= loss0 + 1e-12)

# Rationale: for a smooth convex objective, a small enough LR should reduce the loss.

In [ ]:
# Checks
check('decreased', loss1 <= loss0 + 1e-12)

**Why this works.** for a smooth convex objective, a small enough LR should reduce the loss.